# Explore the grounded-qa starter examples

Select this project's `.venv` as the Python kernel, then run the cells in order. All examples are synthetic. This notebook only reads the data.

Our rule: answer supported parts, identify missing parts with `Insufficient information.`, and use exactly that sentence when nothing requested is supported.

In [1]:
from pathlib import Path

import pandas as pd

# Notebook editors may start in the project folder or notebooks/.
project_root = Path.cwd()
if not (project_root / 'data').is_dir():
    project_root = project_root.parent

data_path = project_root / 'data' / 'source' / 'starter.jsonl'
if not data_path.is_file():
    raise FileNotFoundError('Run this notebook from grounded-qa or its notebooks folder.')

# lines=True reads one JSON object per line into a table.
examples = pd.read_json(data_path, lines=True)
print(f'{len(examples)} questions from {examples["source_id"].nunique()} source records')
examples[['example_id', 'source_id', 'answerability']]

12 questions from 3 source records


,example_id,source_id,answerability
0,workshop_001_q1,workshop_001,answerable
1,workshop_001_q2,workshop_001,answerable
2,workshop_001_q3,workshop_001,partial
3,workshop_001_q4,workshop_001,unanswerable
4,kit_001_q1,kit_001,answerable
5,kit_001_q2,kit_001,answerable
6,kit_001_q3,kit_001,partial
7,kit_001_q4,kit_001,unanswerable
8,ferry_001_q1,ferry_001,answerable
9,ferry_001_q2,ferry_001,answerable


## Inspect one source and its questions

A `source_id` ties questions to the same information. Change `source_to_review` to `kit_001` or `ferry_001` to inspect another group. Check each answer against only the printed information.

In [5]:
source_to_review = 'ferry_001'
source_examples = examples[examples['source_id'] == source_to_review]

if source_examples.empty:
    raise ValueError(f'Unknown source_id: {source_to_review}')

print('INFORMATION:', source_examples.iloc[0]['information'])
for example in source_examples.itertuples(index=False):
    print(f'\n[{example.answerability}] {example.question}')
    print(f'Answer: {example.answer}')

INFORMATION: The fictional Moss Ferry leaves Alder Dock at 09:15 and arrives at Birch Dock at 09:40. A one-way ticket costs 4 credits.

[answerable] Which dock does the Moss Ferry leave from, and at what time?
Answer: Departure dock: Alder Dock. Departure time: 09:15.

[answerable] How much does a one-way ticket on the Moss Ferry cost?
Answer: 4 credits.

[partial] When does the Moss Ferry arrive at Birch Dock, and how many passengers can it carry?
Answer: Arrival time: 09:40. Passenger capacity: Insufficient information.

[unanswerable] What is the name of the Moss Ferry's captain?
Answer: Insufficient information.


## Count the example types in each source group

Each group has two answerable, one partial, and one unanswerable question. When we split the data later, every question in a source group must go to the same split. This prevents the same source information appearing in both training and evaluation.

In [3]:
pd.crosstab(examples['source_id'], examples['answerability'])

answerability,answerable,partial,unanswerable
source_id,,,
ferry_001,2,1,1
kit_001,2,1,1
workshop_001,2,1,1


## Stop and review

Inspect all three source groups before adding more code. For each partial answer, identify the supported fact and the missing fact. These 12 examples are for learning the workflow, not for a meaningful training run. The training and evaluation files remain empty.